# Heme SH Anki builder — Assemble Drive bundles (v0.1)

**What this is for (lazy mode):** one Colab that pulls the ChatGPT/deck-builder files into Google Drive so you can upload folders to ChatGPT without hunting GCS paths.

## What is “TNK”?

**TNK** = **T/NK-cell lymphomas** exemplar package:

`Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip`

It is **not** another raw lecture to convert. It is the **finished style authority**: SOP examples, templates, accepted-tag JSON, shared-back examples, QA patterns, builder scaffolding. Every other Heme SH deck (AML, BM Intro, MDS/MPN, …) should imitate *that* package’s card style.

(Related but different: lecture series `Heme_SH_Hodgkin_T_NK_Cell_1/2` are teaching videos. The TNK zip is the **Anki exemplar** for T/NK lymphomas.)

## What this notebook does

1. Mount Drive + auth GCS (`pathology-annotation-project`)
2. Search Drive for TNK zip + WHO heme JSON (or use paths you set)
3. Download shared SOP/docs from GCS
4. For chosen series: download lecture ZIP(s) + `manifest.json` / `frames.jsonl` / `segments.jsonl` (+ optional chunks)
5. Write a tidy Drive tree you can zip/upload to ChatGPT

**Does not** invent tags. **Does not** attach `tag_audit` / `chunk_audit` / lecture `audit.json`.

GCS shared prefix (already partially published):
`gs://pathology_hub/02_normalized/anki/heme_sh_contextual_cloze_builder_v0_1/`

In [ ]:
# Cell 1 — auth + mounts
!pip -q install google-cloud-storage

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

from google.cloud import storage
from pathlib import Path
import json, re, shutil, zipfile
from datetime import datetime, timezone

PROJECT = "pathology-annotation-project"
HUB = "pathology_hub"
BUILDER_PREFIX = "02_normalized/anki/heme_sh_contextual_cloze_builder_v0_1"
DECK_PREFIX = "02_normalized/lectures/deck_packages"

client = storage.Client(project=PROJECT)
hub = client.bucket(HUB)

def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print("ok", utc_now())

In [ ]:
# Cell 2 — lazy knobs (edit these)

# Where to write on Drive
OUT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/3-Resources/Heme/Anki_Builder_Bundles"),
    Path("/content/drive/MyDrive/3 - Resources/Heme/Anki_Builder_Bundles"),
    Path("/content/drive/MyDrive/Heme_Anki_Builder_Bundles"),
]
OUT_ROOT = next((p for p in OUT_ROOT_CANDIDATES if p.parent.exists()), OUT_ROOT_CANDIDATES[-1])
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Which series to assemble. Use "ALL" or one slug from SERIES_INDEX below.
SERIES_CHOICE = "bm_intro"  # e.g. "bm_intro", "mds_mpn", "aml", or "ALL"

# Speed toggles
DOWNLOAD_LECTURE_ZIPS = True   # False = only sidecars + shared (lighter for ChatGPT)
DOWNLOAD_CHUNKS = True         # optional index only; safe for survey talks with 0 chunks
DOWNLOAD_SEGMENTS = True       # can be large (~0.5–1 MB+ per lecture); set False if slow
UPLOAD_SHARED_AUTHORITY_TO_GCS = False  # True = also push found TNK/WHO up to GCS shared/
MAKE_ZIP_PER_SERIES = True     # creates series_xxx_chatGPT_upload.zip next to the folder

# If auto-find fails, paste absolute Drive paths here:
MANUAL_TNK_ZIP = None  # Path("/content/drive/MyDrive/.../Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip")
MANUAL_WHO_JSON = None  # Path("/content/drive/MyDrive/.../WHO_WHO_JSON_PROCESSED_HEME.json")
MANUAL_SOP_PDF = None   # optional override

print("OUT_ROOT =", OUT_ROOT)
print("SERIES_CHOICE =", SERIES_CHOICE)

In [ ]:
# Cell 3 — series catalog (slug → packages + lecture zip names)
SERIES = {
    "aggressive_b_cell": {
        "title": "Aggressive B-Cell",
        "handoff": "HANDOFF_AGGRESSIVE_B_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_aggressive_b_cell_v0_1"],
        "zips": ["Heme_SH_Aggressive_B_Cell_chatgpt_readable_package.zip"],
    },
    "aml": {
        "title": "AML",
        "handoff": "HANDOFF_AML_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_aml_v0_1"],
        "zips": ["Heme_SH_AML_package.zip"],
    },
    "bm_failure_syndromes": {
        "title": "BM Failure Syndromes",
        "handoff": "HANDOFF_BM_FAILURE_SYNDROMES_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_failure_syndromes_v0_1"],
        "zips": ["Heme_SH_BM_Failure_Syndromes_package.zip"],
    },
    "bm_intro": {
        "title": "BM Intro",
        "handoff": "HANDOFF_BM_INTRO_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_intro_v0_1"],
        "zips": ["Heme_SH_BM_Intro_package.zip"],
        "note": "0 gated chunks — still HIGH YIELD; use transcript+index+frames",
    },
    "bm_systemic_manifestations": {
        "title": "BM Systemic Manifestations",
        "handoff": "HANDOFF_BM_SYSTEMIC_MANIFESTATIONS_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_systemic_manifestations_v0_1"],
        "zips": ["Heme_SH_BM_Systemic_Manifestations_package.zip"],
    },
    "histiocytic": {
        "title": "Histiocytic",
        "handoff": "HANDOFF_HISTIOCYTIC_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_histiocytic_v0_1"],
        "zips": ["Heme_SH_Histiocytic_package.zip"],
    },
    "hodgkin_nlp": {
        "title": "Hodgkin NLP",
        "handoff": "HANDOFF_HODGKIN_NLP_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_nlp_v0_1"],
        "zips": ["Heme_SH_Hodgkin_NLP_package.zip"],
    },
    "hodgkin_overview": {
        "title": "Hodgkin Overview",
        "handoff": "HANDOFF_HODGKIN_OVERVIEW_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_overview_v0_1"],
        "zips": ["Heme_SH_Hodgkin_Overview_package.zip"],
    },
    "hodgkin_t_nk_cell": {
        "title": "Hodgkin T/NK-Cell (parts 1–2)",
        "handoff": "HANDOFF_HODGKIN_T_NK_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_t_nk_cell_1_v0_1", "heme_sh_hodgkin_t_nk_cell_2_v0_1"],
        "zips": [
            "Heme_SH_Hodgkin_T_NK_Cell_1_package.zip",
            "Heme_SH_Hodgkin_T_NK_Cell_2_package.zip",
        ],
    },
    "ia_lpd": {
        "title": "IA-LPD",
        "handoff": "HANDOFF_IA_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_ia_lpd_v0_1"],
        "zips": ["Heme_SH_IA_LPD_package.zip"],
    },
    "ihc_for_lpd": {
        "title": "IHC for LPD",
        "handoff": "HANDOFF_IHC_FOR_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_ihc_for_lpd_v0_1"],
        "zips": ["Heme_SH_IHC_for_LPD_package.zip"],
    },
    "mds_mpn": {
        "title": "MDS/MPN (parts 1–3)",
        "handoff": "HANDOFF_MDS_MPN_ANKI_BUILDER_v0_1.md",
        "packages": [
            "heme_sh_mds_mpn_1_v0_1",
            "heme_sh_mds_mpn_2_v0_1",
            "heme_sh_mds_mpn_3_v0_1",
        ],
        "zips": [
            "Heme_SH_MDS_MPN_1_package.zip",
            "Heme_SH_MDS_MPN_2_package.zip",
            "Heme_SH_MDS_MPN_3_package.zip",
        ],
    },
    "plasma_cell": {
        "title": "Plasma Cell",
        "handoff": "HANDOFF_PLASMA_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_plasma_cell_v0_1"],
        "zips": ["Heme_SH_Plasma_Cell_package.zip"],
    },
    "pt_lpd": {
        "title": "PT-LPD",
        "handoff": "HANDOFF_PT_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_pt_lpd_v0_1"],
        "zips": ["Heme_SH_PT_LPD_package.zip"],
    },
    "reactive_lymphoid_hyperplasia": {
        "title": "Reactive Lymphoid Hyperplasia",
        "handoff": "HANDOFF_REACTIVE_LYMPHOID_HYPERPLASIA_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_reactive_lymphoid_hyperplasia_v0_1"],
        "zips": ["Heme_SH_Reactive_Lymphoid_Hyperplasia_package.zip"],
    },
    "small_b_cell": {
        "title": "Small B-Cell (parts 1–2)",
        "handoff": "HANDOFF_SMALL_B_CELL_ANKI_BUILDER_v0_1.md",
        "packages": [
            "heme_sh_small_b_cell_1_of_2_v0_1",
            "heme_sh_small_b_cell_2_of_2_v0_1",
        ],
        "zips": [
            "Heme_SH_Small_B_Cell_1_of_2_package.zip",
            "Heme_SH_Small_B_Cell_2_of_2_package.zip",
        ],
    },
    "spleen": {
        "title": "Spleen",
        "handoff": "HANDOFF_SPLEEN_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_spleen_v0_1"],
        "zips": ["Heme_SH_Spleen_package.zip"],
    },
}

print(len(SERIES), "series;", ", ".join(SERIES))

In [ ]:
# Cell 4 — find TNK + WHO on Drive (or use MANUAL_* )

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
]

TNK_NAME = "Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip"
WHO_NAME = "WHO_WHO_JSON_PROCESSED_HEME.json"
SOP_GCS = f"{BUILDER_PREFIX}/shared/Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf"

def find_named(name: str, roots):
    hits = []
    for root in roots:
        if not root.exists():
            continue
        # Prefer shallow hits; walk can be slow — cap depth-ish by skipping huge trees
        for p in root.rglob(name):
            if p.is_file():
                hits.append(p)
                if len(hits) >= 5:
                    return hits
    return hits

tnk_path = Path(MANUAL_TNK_ZIP) if MANUAL_TNK_ZIP else None
who_path = Path(MANUAL_WHO_JSON) if MANUAL_WHO_JSON else None
sop_path = Path(MANUAL_SOP_PDF) if MANUAL_SOP_PDF else None

if tnk_path is None:
    hits = find_named(TNK_NAME, SEARCH_ROOTS)
    print("TNK search hits:", hits)
    tnk_path = hits[0] if hits else None

if who_path is None:
    hits = find_named(WHO_NAME, SEARCH_ROOTS)
    # also try common variants
    if not hits:
        for alt in ["WHO_JSON_PROCESSED_HEME.json", "who_who_json_processed_heme.json"]:
            hits = find_named(alt, SEARCH_ROOTS)
            if hits:
                break
    print("WHO search hits:", hits)
    who_path = hits[0] if hits else None

print("TNK =", tnk_path)
print("WHO =", who_path)
if tnk_path is None:
    print("\n⚠️  TNK zip not found on Drive. Put it anywhere under MyDrive or set MANUAL_TNK_ZIP.")
if who_path is None:
    print("\n⚠️  WHO JSON not found on Drive. Put it anywhere under MyDrive or set MANUAL_WHO_JSON.")

In [ ]:
# Cell 5 — helpers: download GCS blob → local path

def gcs_download(blob_name: str, dest: Path, bucket=hub) -> bool:
    blob = bucket.blob(blob_name)
    if not blob.exists():
        print(" MISSING", f"gs://{bucket.name}/{blob_name}")
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == blob.size:
        print(" skip", dest.name, "(exists)")
        return True
    blob.download_to_filename(str(dest))
    print(" got ", dest.name, f"({dest.stat().st_size} bytes)")
    return True

def copy_local(src: Path, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == src.stat().st_size:
        print(" skip", dest.name, "(exists)")
        return
    shutil.copy2(src, dest)
    print(" copy", src.name, "→", dest)

def zip_dir(src_dir: Path, zip_path: Path):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in src_dir.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(src_dir).as_posix())
    print(" zipped", zip_path, f"({zip_path.stat().st_size} bytes)")

# shared folder
SHARED = OUT_ROOT / "_shared"
SHARED.mkdir(parents=True, exist_ok=True)

# SOP from GCS
gcs_download(SOP_GCS, SHARED / "Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf")

# handoffs / prompts / common from GCS docs/
for name in [
    "HANDOFF_HEME_SH_ANKI_BUILDER_COMMON_v0_1.md",
    "HANDOFF_HEME_SH_ANKI_BUILDER_INDEX_v0_1.md",
    "HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md",
]:
    gcs_download(f"{BUILDER_PREFIX}/docs/{name}", SHARED / "docs" / name)

if tnk_path and tnk_path.exists():
    copy_local(tnk_path, SHARED / TNK_NAME)
if who_path and who_path.exists():
    copy_local(who_path, SHARED / WHO_NAME)

# optional: push authority files into GCS so they live with the project
if UPLOAD_SHARED_AUTHORITY_TO_GCS:
    for local_name, dest_name in [
        (TNK_NAME, f"{BUILDER_PREFIX}/shared/{TNK_NAME}"),
        (WHO_NAME, f"{BUILDER_PREFIX}/shared/{WHO_NAME}"),
    ]:
        lp = SHARED / local_name
        if not lp.exists():
            print("skip GCS upload; missing", local_name)
            continue
        blob = hub.blob(dest_name)
        blob.upload_from_filename(str(lp))
        print(" uploaded", f"gs://{HUB}/{dest_name}")

print("SHARED ready:", SHARED)

In [ ]:
# Cell 6 — assemble one or all series into Drive folders (+ optional zip)

slugs = list(SERIES) if SERIES_CHOICE.upper() == "ALL" else [SERIES_CHOICE]
assert all(s in SERIES for s in slugs), f"Unknown slug(s): {slugs}"

SIDECARS = ["manifest.json", "frames.jsonl"]
if DOWNLOAD_SEGMENTS:
    SIDECARS.append("segments.jsonl")
if DOWNLOAD_CHUNKS:
    SIDECARS.append("chunks_indexable.jsonl")

DO_NOT_COPY = {"tag_audit.json", "chunk_audit.json", "audit.json", "segments_indexable.jsonl"}

audit = {
    "schema_version": "heme_anki_drive_bundle_assembler.v0_1",
    "created_at_utc": utc_now(),
    "out_root": str(OUT_ROOT),
    "series": [],
    "shared": {
        "tnk_present": (SHARED / TNK_NAME).exists(),
        "who_present": (SHARED / WHO_NAME).exists(),
        "sop_present": (SHARED / "Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf").exists(),
    },
    "known_limitations": [
        "TNK = T/NK lymphomas Anki STYLE exemplar, not a raw lecture.",
        "chunks_indexable is navigation only; BM Intro may have 0 chunks and is still high-yield.",
        "tag_audit/chunk_audit/audit.json intentionally excluded.",
    ],
}

for slug in slugs:
    meta = SERIES[slug]
    series_dir = OUT_ROOT / f"series_{slug}"
    # refresh folder lightly
    series_dir.mkdir(parents=True, exist_ok=True)

    # pointer copies of shared authority into each series (ChatGPT-friendly one-folder uploads)
    style = series_dir / "00_style_and_who"
    style.mkdir(exist_ok=True)
    for fname in [
        TNK_NAME,
        WHO_NAME,
        "Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf",
    ]:
        src = SHARED / fname
        if src.exists():
            copy_local(src, style / fname)

    # docs
    docs_dir = series_dir / "01_docs"
    docs_dir.mkdir(exist_ok=True)
    for name in [
        "HANDOFF_HEME_SH_ANKI_BUILDER_COMMON_v0_1.md",
        "HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md",
        meta["handoff"],
    ]:
        # series handoff from GCS docs/
        if name == meta["handoff"]:
            gcs_download(f"{BUILDER_PREFIX}/docs/{name}", docs_dir / name)
        else:
            src = SHARED / "docs" / name
            if src.exists():
                copy_local(src, docs_dir / name)
            else:
                gcs_download(f"{BUILDER_PREFIX}/docs/{name}", docs_dir / name)

    # lectures
    for i, (pkg, zname) in enumerate(zip(meta["packages"], meta["zips"]), start=1):
        lec = series_dir / f"lecture_{i}"
        lec.mkdir(exist_ok=True)
        if DOWNLOAD_LECTURE_ZIPS:
            gcs_download(zname, lec / zname)
        for sc in SIDECARS:
            gcs_download(f"{DECK_PREFIX}/{pkg}/{sc}", lec / sc)

    # README for humans / ChatGPT
    note = meta.get("note", "")
    readme = f"""# {meta['title']} — ChatGPT Anki builder bundle\n\n"""
    readme += "## TNK reminder\n"
    readme += "`00_style_and_who/` contains the **T/NK lymphomas contextual-cloze exemplar** (style law + accepted tags).\n"
    readme += "It is not the lecture you are converting.\n\n"
    readme += "## Upload to ChatGPT\n"
    readme += "1. Prefer the zip sibling `series_<slug>_chatGPT_upload.zip` if created.\n"
    readme += "2. Or upload this whole folder.\n"
    readme += "3. Paste the prompt from `01_docs/HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md`.\n\n"
    if note:
        readme += f"## Series note\n{note}\n\n"
    readme += "## Do not use\n"
    readme += "tag_audit.json, chunk_audit.json, audit.json, or frame/chunk primary_tag as tag authority.\n"
    (series_dir / "README_FOR_CHATGPT.md").write_text(readme)

    zip_path = None
    if MAKE_ZIP_PER_SERIES:
        zip_path = OUT_ROOT / f"series_{slug}_chatGPT_upload.zip"
        zip_dir(series_dir, zip_path)

    audit["series"].append({
        "slug": slug,
        "title": meta["title"],
        "dir": str(series_dir),
        "zip": str(zip_path) if zip_path else None,
        "packages": meta["packages"],
    })
    print("DONE", slug, "→", series_dir)

audit_path = OUT_ROOT / f"assemble_audit_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
audit_path.write_text(json.dumps(audit, indent=2))
print("\nAUDIT", audit_path)
print(json.dumps(audit["shared"], indent=2))
print("\nNext: open Drive →", OUT_ROOT)
print("Upload series_*_chatGPT_upload.zip to ChatGPT + paste prompt from HANDOFF_CHATGPT_HEME_ANKI_PROMPTS.")

## Lazy recipe

1. Put (once) on Drive somewhere under MyDrive:
   - `Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip`
   - `WHO_WHO_JSON_PROCESSED_HEME.json`
2. Set `SERIES_CHOICE = "bm_intro"` (or `"ALL"` if patient).
3. Runtime → Run all.
4. Download / open `series_bm_intro_chatGPT_upload.zip` from Drive.
5. In ChatGPT: attach that zip + paste the BM Intro block from `HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md`.

Optional: set `UPLOAD_SHARED_AUTHORITY_TO_GCS = True` once TNK/WHO are found — lands them in the project GCS shared folder forever.